# Change in mean

The most common change-detection problem is detecting shifts in the mean of a time series. Skchange offers several scorers for this feature; the two most useful entry points are `L2Cost` (paired with `PELT`) for exact optimisation, and `CUSUM` (paired with `SeededBinarySegmentation`) for a faster approximate search.

In [ ]:
import plotly.io as pio

from skchange.new_api.datasets import generate_piecewise_normal_data
from skchange.new_api.utils.plotting import plot_detections

pio.renderers.default = "notebook"

X = generate_piecewise_normal_data(
    means=[0, 10, 0, -3, 5, 1],
    lengths=[30, 5, 15, 50, 60, 40],
    seed=0,
)

The series has six segments of unit-variance Gaussian data with five underlying changepoints at indices 30, 35, 50, 100, and 160. The second segment is a short spike that is easy to miss with a coarse search.

## `PELT` and `L2Cost`

`PELT` minimises the total sum of segment costs plus a per-changepoint penalty. With `L2Cost` — the sum of squared deviations from each segment's sample mean — this recovers all five changepoints on the shared `X`.

In [ ]:
from skchange.new_api.detectors import PELT
from skchange.new_api.interval_scorers import L2Cost

detector = PELT(L2Cost(), penalty=10.0)
changepoints = detector.fit_predict(X)

plot_detections(X, changepoints=changepoints).show()
print(changepoints)

## `SeededBinarySegmentation` and `CUSUM`

For long series, an exact `PELT` search can become expensive. `SeededBinarySegmentation` evaluates a change score on a pre-computed grid of intervals and picks the local maxima that exceed the penalty. Paired with the classical `CUSUM` statistic for a change in mean, it recovers the same changepoints at a fraction of the cost.

In [ ]:
from skchange.new_api.detectors import SeededBinarySegmentation
from skchange.new_api.interval_scorers import CUSUM

detector = SeededBinarySegmentation(CUSUM(), min_subinterval_length=2, penalty=5.0)
changepoints = detector.fit_predict(X)

plot_detections(X, changepoints=changepoints).show()
print(changepoints)

Note the `min_subinterval_length=2`: with the default of 5, the two-sample-wide spike segment `[30, 35)` would fall through the grid.

## Which one to pick

- **Use `PELT` + `L2Cost`** when you want an optimal segmentation of the whole series and the sample size is manageable. The output is the segmentation that minimises the penalised total squared error.
- **Use `SeededBinarySegmentation` + `CUSUM`** when speed matters or the series is very long. The output is a set of local maxima of the CUSUM statistic; not globally optimal, but usually very close and much faster to compute.

For guidance on setting the penalty, see the [Penalties section of the concepts page](../concepts.ipynb#Penalties) and the tuning material referenced there.